In [22]:
import os
import random
import string
import math

import hail as hl
from ukb_utils import hail_init
from ko_utils import ko
from ko_utils import io
from ko_utils import samples
from ukb_utils import genotypes

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/for_stefania/')
hail_init.hail_bmrc_init('hail.log', 'GRCh37')

Running on Apache Spark version 3.1.2
SparkUI available at http://compa033.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to hail.log


In [3]:
HAPLOTYPES="/well/lindgren-ukbb/projects/ukbb-11867/stefania/IBD_v2/WBsib/ibdseg/chr21"
IMPUTED="data/ukb_imp_chr21_v3.bgen"
SAMPLES="/gpfs1/well/lindgren/UKBIOBANK/DATA/SAMPLE_FAM/ukb11867_imp_chr1_v3_s487395.sample"

In [4]:
#hl.index_bgen(IMPUTED ,contig_recoding={"22": "22"}, reference_genome='GRCh37')

2022-06-08 15:44:25 Hail: INFO: Finished writing index file for file:/gpfs3/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/for_stefania/data/ukb_imp_chr21_v3.bgen
2022-06-08 15:44:25 Hail: INFO: Number of BGEN files indexed: 1


In [5]:
imp = hl.import_bgen(IMPUTED, entry_fields=['GT'], sample_file=SAMPLES)

2022-06-08 15:44:26 Hail: INFO: Number of BGEN files parsed: 1
2022-06-08 15:44:26 Hail: INFO: Number of samples in BGEN files: 487409
2022-06-08 15:44:26 Hail: INFO: Number of variants across all BGEN files: 1261158


In [25]:
geno = genotypes.get_ukb_genotypes_bed(chroms=[21])

2022-06-09 11:13:46 Hail: INFO: Found 488377 samples in fam file.
2022-06-09 11:13:46 Hail: INFO: Found 11342 variants in bim file.


In [6]:
hap = io.import_table(HAPLOTYPES,"plink")

2022-06-08 15:44:28 Hail: INFO: Found 34608 samples in fam file.
2022-06-08 15:44:28 Hail: INFO: Found 9791 variants in bim file.


In [7]:
hap.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'fam_id': str
    'pat_id': str
    'mat_id': str
    'is_female': bool
    'is_case': bool
----------------------------------------
Row fields:
    'locus': locus<GRCh37>
    'alleles': array<str>
    'rsid': str
    'cm_position': float64
    'info': struct {
        AC: int32, 
        AF: float64, 
        AN: int32, 
        homozygote_count: array<int32>
    }
----------------------------------------
Entry fields:
    'GT': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [8]:
imp.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh37>
    'alleles': array<str>
    'rsid': str
    'varid': str
----------------------------------------
Entry fields:
    'GT': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [9]:
mt = hap.annotate_entries(GT_imp = imp[hap.row_key, hap.col_key].GT)

2022-06-08 15:45:16 Hail: WARN: cols(): Resulting column table is sorted by 'col_key'.
    To preserve matrix table column order, first unkey columns with 'key_cols_by()'


In [ ]:
mt.describe()

In [10]:
io.export_table(mt, "derived/220608_ukb_hap_chr21_v2_AND_ukb_imp_chr21_v3", "mt")


Writing to MT: derived/220608_ukb_hap_chr21_v2_AND_ukb_imp_chr21_v3.mt



2022-06-08 18:11:15 Hail: INFO: wrote matrix table with 9791 rows and 34608 columns in 6 partitions to derived/220608_ukb_hap_chr21_v2_AND_ukb_imp_chr21_v3.mt
    Total size: 81.85 MiB
    * Rows/entries: 81.52 MiB
    * Columns: 335.21 KiB
    * Globals: 11.00 B
    * Smallest partition: 1632 rows (13.06 MiB)
    * Largest partition:  1632 rows (14.43 MiB)


In [38]:
mt = io.import_table("derived/220608_ukb_hap_chr21_v2_AND_ukb_imp_chr21_v3.mt",'mt')

In [39]:
#mt.describe()

In [40]:
mt = mt.annotate_entries(GT_calls = geno[mt.row_key, mt.col_key].GT)

In [41]:
mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'fam_id': str
    'pat_id': str
    'mat_id': str
    'is_female': bool
    'is_case': bool
----------------------------------------
Row fields:
    'locus': locus<GRCh37>
    'alleles': array<str>
    'rsid': str
    'cm_position': float64
    'info': struct {
        AC: int32, 
        AF: float64, 
        AN: int32
    }
----------------------------------------
Entry fields:
    'GT': call
    'GT_imp': call
    'GT_calls': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [42]:
io.export_table(mt, "derived/220608_ukb_hap_chr21_v2_AND_ukb_imp_chr21_v3_AND_calls_new", "mt")


Writing to MT: derived/220608_ukb_hap_chr21_v2_AND_ukb_imp_chr21_v3_AND_calls_new.mt



2022-06-09 12:20:16 Hail: INFO: wrote matrix table with 9791 rows and 34608 columns in 6 partitions to derived/220608_ukb_hap_chr21_v2_AND_ukb_imp_chr21_v3_AND_calls_new.mt


In [43]:
mt = io.import_table("derived/220608_ukb_hap_chr21_v2_AND_ukb_imp_chr21_v3_AND_calls_new.mt",'mt')

In [44]:
mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'fam_id': str
    'pat_id': str
    'mat_id': str
    'is_female': bool
    'is_case': bool
----------------------------------------
Row fields:
    'locus': locus<GRCh37>
    'alleles': array<str>
    'rsid': str
    'cm_position': float64
    'info': struct {
        AC: int32, 
        AF: float64, 
        AN: int32
    }
----------------------------------------
Entry fields:
    'GT': call
    'GT_imp': call
    'GT_calls': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [45]:
mt.entries().flatten().export("derived/220608_ukb_hap_chr21_v2_AND_ukb_imp_chr21_v3_AND_calls_new.txt.gz")

2022-06-09 16:29:51 Hail: INFO: merging 6 files totalling 1.7G...
2022-06-09 16:29:53 Hail: INFO: while writing:
    derived/220608_ukb_hap_chr21_v2_AND_ukb_imp_chr21_v3_AND_calls_new.txt.gz
  merge time: 2.767s


In [ ]:
mt = hl.import_vcf("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/data/conditional/rare/combined/ukb_eur_wes_200k_chr21_maf0to5e-2_pLoF_damaging_missense.vcf.bgz")

In [ ]:
mt.describe()

In [ ]:
mt = mt.annotate_rows(ds_mean = hl.agg.stats(mt.DS).mean)

In [ ]:
mt.filter_rows(mt.ds_mean <= 0).count()
mt.filter_rows(mt.ds_mean >= 1).count()

In [ ]:
mt = mt.annotate_rows(ds_stdev = hl.agg.stats(mt.DS).stdev)
mt.filter_rows(mt.ds_stdev <= 0).count()